In [1]:
import os
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import sentencepiece as spm
from collections import Counter

# import personalized classes and configs
from configs import corpus, tok_prefix, model_config
from transformer import MultiLangDataset, TransformerModel, create_causal_mask

In [2]:
if not os.path.exists("tokenizers"):
    raise FileNotFoundError(f"The tokenizers directory does not exist. Make sure you run the training script first.")

if not os.path.exists("results"):
    raise FileNotFoundError(f"The results directory does not exist. Make sure you run the training script first.")

if not os.path.exists("models"):
    raise FileNotFoundError(f"The models directory does not exist. Make sure you run the training script first.")

In [3]:
# make sure GPU is used
if not torch.cuda.is_available():
    raise RuntimeError("no GPU available")

device = torch.device('cuda:0')

In [4]:
#################################################
# Comparing BPC between models
#################################################

def load_model(name: str, tokenizer: spm.SentencePieceProcessor) -> TransformerModel:
    """
    Load the specified model from weights computed in training.

    :name: name of the tokenizer the model was trained with
    :tokenizer: the loaded tokenizer model
    
    return: Transformer model
    """
    model = TransformerModel(
        num_layers=model_config["num_layers"],
        num_heads=model_config["num_heads"],
        hidden_dim=model_config["hidden_dim"],
        ff_dim=model_config["ff_dim"],
        seq_len=model_config["context_len"],
        vocab_size=tokenizer.get_piece_size(),
        device=device,
        dropout=model_config["dropout"]
    ).to(device)

    path = f"models/{name}_model_epochs{model_config["epochs"]}_batchsize{model_config["batch_size"]}.pth"

    if not os.path.exists(path):
        raise FileNotFoundError(f"Model {name} not found in {path}")

    # load weights into the model initialized above
    model.load_state_dict(torch.load(path, map_location=device))
    # set to evaluation mode
    model.eval()

    return model
    

def batch_nll(model: TransformerModel, X: torch.tensor, y: torch.tensor) -> float:
    """
    Negative log-likelihood (units: nats) of y given X for one batch.

    Method:
    1. get log probability of next token using log_softmax
    2. take the negative of the log prob to make the loss positive
    3. add all the log-probs up

    :model: trained LM 
    :X: input token ids
    :y: target token ids (shifted once to the right wrt X)

    return: total loss for the batch
    """
    X = X.to(device)
    y = y.to(device)

    # apply mask so each position in X only sees previous tokens
    mask = create_causal_mask(X.shape[1], device)
    # logit shape: (batch_size, seq_len, vocab_size)
    logits = model(X, mask.unsqueeze(0))

    vocab_size = logits.shape[-1]

    # step 1
    log_probs = F.log_softmax(logits, dim=-1)
    log_probs = log_probs.reshape(-1, vocab_size) # shape ( batch_size * seq_len, vocab_size)
    golden = y.reshape(-1) # shape (batch_size * seq_len, )

    # step 2 + 3
    # since they are logs now the probabilities are added instead of multiplied
    nll = F.nll_loss(log_probs, golden, reduction="sum")

    return nll.item()


def total_nll(model: TransformerModel, tokenizer: spm.SentencePieceProcessor, sents: list[str] , seq_len: int, batch_size: int) -> tuple[float, int, int]:
    """
    Total negative log-liklihoods (unit: nats) for every token in the sentences.

    :sents: list of sentences in on language
    :seq_len: context window length
    :batch_size: number of context windows per batch

    return: (total prob, total number of tokens, number of unknown tokens)
    """
    with torch.no_grad():
        # load dataset (tokenizes and batches the corpus)
        dataset = MultiLangDataset(sents, tokenizer, seq_len)
        loader = DataLoader(dataset, batch_size, shuffle=False)
    
        # total nll of every token 
        nll = 0.0

        # loop through data and sum the nll from the batches
        for X, y in loader:
            nll += batch_nll(model, X, y)

        # find out if there was data at the end that didn't fit into a window
        start = len(dataset) * seq_len
        tail = dataset.encoded[start:]
        # if there is data remaining, add those probs to a batch
        if len(tail) > 1:
            nll += batch_nll(model, tail[:-1].unsqueeze(0), tail[1:].unsqueeze(0))

        # count unknown tokens in the dataset
        ids = dataset.encoded
        unk_count = 0
        for token_id in ids.tolist():
            if token_id == tokenizer.unk_id():
                
                unk_count += 1
    
    return nll, len(ids), unk_count

In [5]:
# load corpus texts (only test split)
text = {}
for lang, file in corpus["test"].items():
    with open(file, encoding="utf-8") as f:
        # each sentence is its own string
        text[lang] = f.readlines()
    

In [6]:
results = {}

# loop through tokenizer models
for name in tok_prefix.keys():
    # load the current pre-trained model
    tokenizer = spm.SentencePieceProcessor(model_file=tok_prefix[name] + ".model")
    # load the transformer model for that tokenizer
    model = load_model(name, tokenizer)

    # initialize empty results
    results[name] = {}
    all_nll = 0.0
    all_chars = 0

    # loop through each language and evaluate the transformer on it
    for lang, sents in text.items():
        nll, num_tokens, num_unk = total_nll(model, tokenizer, sents, model_config["context_len"], model_config["batch_size"])

        # calculate the number of chars in the current text split
        tot_chars = 0
        for s in sents:
            tot_chars += len(s.strip())

        # apply BPC equation from assignment instructions
        bpc = nll / (tot_chars * math.log(2))
        # char per token formula from ___ todo
        chars_per_token = tot_chars / (num_tokens - 2 * len(sents))
        
        # save results
        results[name][lang] = {
            "BPC" : bpc,
            "chars_per_token": chars_per_token,
            "unk": num_unk
        }

        # add these statistics overall count 
        all_nll += nll
        all_chars += tot_chars

    # overall BPC for all languages
    results[name]["overall"] = {
        "BPC": all_nll / (all_chars * math.log(2))
    }

    
    

In [7]:
# print results

for model in results:
    print("=========")
    print(f"Model: {model}")

    # loop through each language and print results
    for lang in results[model]:
        if lang == "overall":
            print(f"Overall BPC: {results[model]['overall']['BPC']:.3f}")
        else:
            print(f"* Language: {lang}")
            print(f"\tBPC: {results[model][lang]['BPC']:.3f}")
            print(f"\tCharacters per token: {results[model][lang]['chars_per_token']:.3f}")
            print(f"\tNumber of unknown tokens: {results[model][lang]['unk']}")
        print()
    

Model: char_level
* Language: en
	BPC: 2.185
	Characters per token: 0.988
	Number of unknown tokens: 6

* Language: tr
	BPC: 2.221
	Characters per token: 0.990
	Number of unknown tokens: 6

* Language: zh
	BPC: 6.479
	Characters per token: 0.975
	Number of unknown tokens: 249

Overall BPC: 3.628

Model: bpe_small
* Language: en
	BPC: 2.104
	Characters per token: 2.199
	Number of unknown tokens: 6

* Language: tr
	BPC: 2.158
	Characters per token: 2.125
	Number of unknown tokens: 6

* Language: zh
	BPC: 6.536
	Characters per token: 1.067
	Number of unknown tokens: 249

Overall BPC: 3.599

Model: bpe_large
* Language: en
	BPC: 2.071
	Characters per token: 3.464
	Number of unknown tokens: 6

* Language: tr
	BPC: 2.150
	Characters per token: 3.514
	Number of unknown tokens: 6

* Language: zh
	BPC: 6.616
	Characters per token: 1.338
	Number of unknown tokens: 249

Overall BPC: 3.612



In [12]:
##########################
# Who gets the vocabulary?
##########################

def count_tokens_per_lang(tokenizer: spm.SentencePieceProcessor, split: str) -> dict:
    """
    Frequency of each token id in each language.

    :tokenizer: tokenizer model
    :split: data split (train, valid, test)

    return: dictionary of where key:language, value: counter object for token id counts
    """

    counts = {}
    # loop through each language split
    for lang, file in corpus[split].items():
        with open(file, encoding="utf-8") as f:
            lines = f.readlines()
        lang_counter = Counter()
        # loop through each sentence in the current split. 
        # count the number of times each ID appears
        for sent_ids in tokenizer.encode(lines):
            lang_counter.update(sent_ids)
        counts[lang] = lang_counter
    return counts

def label_tokens(tokenizer: spm.SentencePieceProcessor, counts: int) -> dict:
    """
    Identify which language each token belongs to. 
    Same formula from assignment instructions:
    p(L | t) = count(t, L) / sum over L'[count(t, L')]

    Threshold of 90% to be considered belonging to a language.
    
    Label options:
        - en
        - tr
        - zh
        - en+tr
        - en+zh
        - tr+zh
        - shared (all three)
        - unused (none)

    :tokenizer:
    :counts: dict of token count per language
    
    return: dict of key: token_id, value: label
    """

    # loop through the token ids in the tokenizer and count the number
    # of times each token appears in each language
    labels = {}
    for token_id in range(tokenizer.get_piece_size()):
        # skip unk, bos and eos tokens
        if tokenizer.is_control(token_id) or tokenizer.is_unknown(token_id):
            continue

        # freq of this token in each language
        per_lang_count = {}
        for lang in counts:
            per_lang_count[lang] = counts[lang][token_id]

        curr_token_total = sum(per_lang_count.values())

        # if token freq is zero across all three languages
        # token is unused
        if curr_token_total == 0:
            labels[token_id] = "unused"
            continue

        # calculate p(L | t) for each language
        p = {}
        for lang in per_lang_count:
            # (count token in current lang) ÷ (count token in all languages)
            p[lang] = per_lang_count[lang]/curr_token_total

        # find the most probable language for the given token
        best_lang = max(p, key=p.get) 
        # if the language has a probability over the threshold, it is considered to only belong to that language
        if p[best_lang] >= threshold: 
            labels[token_id] = best_lang
        else:
            # find out which languages share the token
            lowest_lang = min(p, key=p.get)
            # if the least probable language has a prob that is small enough
            # we know that the token is just shared between the remaining two languages
            if p[lowest_lang] < 1 - threshold:
                # the other two languages share the token
                pair = []
                for lang in ["en", "tr", "zh"]:
                    if lang != lowest_lang:
                        pair.append(lang)
                labels[token_id] = pair[0] + "+" + pair[1]
            else:
                # all three languages use the token
                labels[token_id] = "shared"

    return labels 

def is_merge(tokenizer: spm.SentencePieceProcessor, token_id: int) -> bool:
    """
    Check if token is longer than one character. If so, it is the result of a BPE merge.

    :tokenizer:
    :token_id:

    return: true if token is a merge, false if not
    """
    # use token id to get the token
    # remove underscores (does not count as a char)
    piece = tokenizer.id_to_piece(token_id).replace("▁", "")
    if len(piece) > 1:
        return True
        
    return False



In [13]:
# analyze shared tokens BPE tokenizers
# no merges happen for the character-level tokenizer, but vocabulary sizes are still interesting

threshold = 0.9

all_labels = {}
# loop through tokenizer models
for name in tok_prefix.keys():
    # load the current pre-trained model
    tokenizer = spm.SentencePieceProcessor(model_file=tok_prefix[name] + ".model")
    # count on the training set since that is the set the vocabs were built on
    # and we don't want to count unkowns here
    counts = count_tokens_per_lang(tokenizer, "train")
    labels = label_tokens(tokenizer, counts)
    all_labels[name] = (tokenizer, labels)

    all_counts = Counter(labels.values())
    merge_counts = Counter()
    for token_id, label in labels.items():
        if is_merge(tokenizer, token_id):
            merge_counts[label] += 1

    print("==========")
    print(name)
    print("all vocab items:", dict(all_counts))
    print("learned merges:", dict(merge_counts))

# find where extra merges were allocated
small_tok, small_labels = all_labels["bpe_small"]
large_tok, large_labels = all_labels["bpe_large"]

small_pieces = set()
for token_id in range(small_tok.get_piece_size()):
    piece = small_tok.id_to_piece(token_id)
    small_pieces.add(piece)

new_labels = Counter()
for token_id, label in large_labels.items():
    if large_tok.id_to_piece(token_id) not in small_pieces:
        new_labels[label] += 1

print("tokens only in large vocab: ", dict(new_labels))

char_level
all vocab items: {'shared': 176, 'en+tr': 56, 'tr': 117, 'zh': 8716, 'en': 89, 'en+zh': 114, 'tr+zh': 60}
learned merges: {}
bpe_small
all vocab items: {'en+tr': 324, 'en': 193, 'tr': 312, 'en+zh': 131, 'zh': 8766, 'shared': 210, 'tr+zh': 61}
learned merges: {'en+tr': 230, 'en': 104, 'tr': 188, 'en+zh': 18, 'zh': 41, 'shared': 18}
bpe_large
all vocab items: {'shared': 1009, 'en+tr': 1434, 'en': 1730, 'tr': 2679, 'en+zh': 517, 'zh': 12483, 'tr+zh': 127, 'unused': 18}
learned merges: {'shared': 785, 'en+tr': 1364, 'en': 1640, 'tr': 2553, 'en+zh': 395, 'zh': 3630, 'tr+zh': 63, 'unused': 16}
tokens only in large vocab:  {'tr': 2378, 'en': 1573, 'zh': 3717, 'en+zh': 361, 'en+tr': 1183, 'shared': 711, 'tr+zh': 61, 'unused': 16}


In [10]:
#######################
# Generating Sentences
#######################

def generate_text(model: TransformerModel, tokenizer: spm.SentencePieceProcessor, beginning: str, max_len: int = 350, temp: float = 0.1) -> str:
    model.eval()

    # Encode the beginning, provided to the function
    input_ids = torch.tensor([tokenizer.bos_id()] + tokenizer.encode(beginning)).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            context = input_ids[:, -model_config["context_len"]:]
            mask = create_causal_mask(context.shape[1], device)

            # get model predictions for next token
            outputs = model(context, mask.unsqueeze(0))
            next_token_logits = outputs[:, -1, :]/ temp
            # sample from the distribution
            probs = F.softmax(next_token_logits, dim=1)
            next_token = torch.multinomial(probs, num_samples=1)

            # append to input ids
            input_ids = torch.cat([input_ids, next_token], dim=1)

            # stop if we predict the end token
            if next_token[0].item() == tokenizer.eos_id():
                break

    return tokenizer.decode(input_ids[0].tolist())



In [11]:
# generate sentences 

# I manually selected the sentences in the sents list from the test data
sents = [
    "Siegfried kills Mime by chopping his head off with the sword.",
    "Oklahoma City is the state's capital and largest city by population and land area.",
    "The troubles stemmed from the cultural clash between the tribal groups and the plains people.",
    
    "Rivayete göre kırklar mevkiinden geçen gelin buradaki kavak ağaçlarının Tanrıya secde ettiklerini görür.",
    "Bundan dolayı alternatif diyotlar verimli güç dönüşümü için kullanılmak zorundadırlar.",
    "Dünya Gümrük Örgütü bünyesinde imzalanan bazı uluslararası sözleşmelere Türkiye de taraf olmuştur.",

    "广场公园共分為始绿园、春之园、感觉园、嗅觉园、地质园、触觉园、干河园、芳草园、视觉园、听觉园、味觉园、自然生态园、音乐园、大树主题广场、市民林等区域。",
    "，并获封三等承恩公，此宅遂升格为承恩公府。",
    "，“玛夏（Masher）”则有可能来源于mashadar（意为前镗枪）或者masha（意为女王）。"
]


# loop through tokenizer models
for name in tok_prefix.keys():
    # load the current pre-trained model
    tokenizer = spm.SentencePieceProcessor(model_file=tok_prefix[name] + ".model")
    # load the transformer model for that tokenizer
    model = load_model(name, tokenizer)

    print("=============")
    print(name)
    for sent in sents:
        half = len(sent) // 2
        prompt = sent[:half]
        print(f"golden: {sent}")
        print(f"prompt: {prompt}")
        output = generate_text(model, tokenizer, prompt, max_len=100)
        print(f"model output: {output}")
        print()


char_level
golden: Siegfried kills Mime by chopping his head off with the sword.
prompt: Siegfried kills Mime by choppi
model output: Siegfried kills Mime by chopping the state of the country.

golden: Oklahoma City is the state's capital and largest city by population and land area.
prompt: Oklahoma City is the state's capital and 
model output: Oklahoma City is the state's capital and the country.

golden: The troubles stemmed from the cultural clash between the tribal groups and the plains people.
prompt: The troubles stemmed from the cultural clash b
model output: The troubles stemmed from the cultural clash be a state of the second of the state of the second of the second of the state of the state of the co

golden: Rivayete göre kırklar mevkiinden geçen gelin buradaki kavak ağaçlarının Tanrıya secde ettiklerini görür.
prompt: Rivayete göre kırklar mevkiinden geçen gelin buradak
model output: Rivayete göre kırklar mevkiinden geçen gelin buradaki bir karşılaştırmaktadır.

golden: B